# Notebook 4 — PPO Training with Checkpoints

Trains PPO on `ALE/Pong-v5` using 8 parallel environments.
PPO collects `n_steps` per env before each update, giving a large diverse minibatch.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
except ImportError:
    SAVE_DIR = 'models'

import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import gymnasium as gym
import ale_py
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

gym.register_envs(ale_py)

ENV_ID          = 'ALE/Pong-v5'
RUN_NAME        = 'ppo_pong_default'
N_ENVS          = 8
TOTAL_TIMESTEPS = 2_000_000

In [ ]:
vec_env  = make_atari_env(ENV_ID, n_envs=N_ENVS, seed=42)
vec_env  = VecFrameStack(vec_env, n_stack=4)

eval_env = make_atari_env(ENV_ID, n_envs=1, seed=0)
eval_env = VecFrameStack(eval_env, n_stack=4)

run_dir = os.path.join(SAVE_DIR, RUN_NAME)

checkpoint_cb = CheckpointCallback(
    save_freq=max(100_000 // N_ENVS, 1),
    save_path=run_dir,
    name_prefix='ppo',
)
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(run_dir, 'best'),
    log_path=os.path.join(run_dir, 'eval_logs'),
    eval_freq=max(50_000 // N_ENVS, 1),
    n_eval_episodes=20,
    deterministic=True,
)

In [ ]:
model = PPO(
    'CnnPolicy', vec_env,
    learning_rate=2.5e-4,
    n_steps=128,        # steps per env before update — total batch = 128 * 8 = 1024
    batch_size=256,     # minibatch size during SGD
    n_epochs=4,         # passes over the collected rollout
    gamma=0.99,
    gae_lambda=0.95,    # GAE smoothing between Monte Carlo and TD
    clip_range=0.1,     # PPO clip — prevents too-large policy updates
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    tensorboard_log=os.path.join(SAVE_DIR, 'tensorboard'),
    verbose=1,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[checkpoint_cb, eval_cb],
    tb_log_name=RUN_NAME,
)

model.save(os.path.join(run_dir, 'ppo_final'))
vec_env.close()
eval_env.close()
print('Done!')

In [ ]:
import glob
ckpts = sorted(glob.glob(os.path.join(run_dir, 'ppo_*.zip')))
print(f'Saved checkpoints ({len(ckpts)}):')
for c in ckpts:
    print(' ', c)

## Hyperparameter Experiment — Smaller Batch

Re-run with `batch_size=64` and `n_envs=4` to compare data efficiency.
A smaller batch can sometimes find better policies but is noisier.

In [ ]:
RUN_NAME_2 = 'ppo_pong_small_batch'
N_ENVS_2   = 4

vec_env2  = make_atari_env(ENV_ID, n_envs=N_ENVS_2, seed=42)
vec_env2  = VecFrameStack(vec_env2, n_stack=4)

run_dir2 = os.path.join(SAVE_DIR, RUN_NAME_2)
checkpoint_cb2 = CheckpointCallback(
    save_freq=max(100_000 // N_ENVS_2, 1),
    save_path=run_dir2,
    name_prefix='ppo',
)

model2 = PPO(
    'CnnPolicy', vec_env2,
    learning_rate=2.5e-4,
    n_steps=128,
    batch_size=64,    # smaller minibatch
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    tensorboard_log=os.path.join(SAVE_DIR, 'tensorboard'),
    verbose=1,
)

model2.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_cb2,
    tb_log_name=RUN_NAME_2,
)
model2.save(os.path.join(run_dir2, 'ppo_final'))
vec_env2.close()
print('Experiment 2 done!')